In [0]:
# %sql
# WITH CUSTOMER_PUCHASE_DATE AS (
#   SELECT
#     A.CUSTOMER_ID,
#     A.FIRST_NAME,
#     A.LAST_NAME,
#     MIN(B.ORDER_DATE_UPDATED) AS FIRST_ORDER_DATE,
#     MAX(B.ORDER_DATE_UPDATED) AS LAST_ORDER_DATE,
#     DATEDIFF(CURRENT_DATE(), MAX(B.ORDER_DATE_UPDATED)) AS DAYS_SINCE_LAST_ORDER
#   FROM
#     ECOMMERCE.SILVER.CUSTOMERS A
#       INNER JOIN ECOMMERCE.SILVER.ORDERS B
#         ON A.CUSTOMER_ID = B.CUSTOMER_ID
#   GROUP BY
#     1,
#     2,
#     3
# ),
# SALES_KPIS AS (
#   SELECT
#     A.CUSTOMER_ID,
#     SUM(
#       CASE
#         WHEN A.ORDER_STATUS = 'DELIVERED' THEN B.NET_AMOUNT
#         ELSE 0
#       END
#     ) AS TOTAL_SPEND,
#     COUNT(DISTINCT
#       CASE
#         WHEN A.ORDER_STATUS = 'DELIVERED' THEN A.ORDER_ID
#         ELSE NULL
#       END
#     ) AS TOTAL_ORDERS,
#     SUM(
#       CASE
#         WHEN A.ORDER_STATUS = 'DELIVERED' THEN B.QUANTITY
#         ELSE 0
#       END
#     ) AS TOTAL_QUANTITY
#   FROM
#     ECOMMERCE.SILVER.ORDERS A
#       INNER JOIN ECOMMERCE.SILVER.ORDER_ITEMS B
#         ON A.ORDER_ID = B.ORDER_ID
#   GROUP BY
#     1
# ),
# PREFERRED_PRODUCTS AS (
#   SELECT
#     CUSTOMER_ID,
#     PRODUCT_NAME,
#     PURCHASE_FREQUENCY
#   FROM
#     (
#       SELECT
#         A.CUSTOMER_ID,
#         B.PRODUCT_ID,
#         C.PRODUCT_NAME,
#         COUNT(DISTINCT A.ORDER_ID) AS PURCHASE_FREQUENCY,
#         ROW_NUMBER() OVER (
#             PARTITION BY A.CUSTOMER_ID
#             ORDER BY COUNT(DISTINCT A.ORDER_ID) DESC
#           ) AS RN
#       FROM
#         ECOMMERCE.SILVER.ORDERS A
#           INNER JOIN ECOMMERCE.SILVER.ORDER_ITEMS B
#             ON A.ORDER_ID = B.ORDER_ID
#           INNER JOIN ECOMMERCE.SILVER.PRODUCTS C
#             ON B.PRODUCT_ID = C.PRODUCT_ID
#       WHERE
#         A.ORDER_STATUS = 'DELIVERED'
#       GROUP BY
#         1,
#         2,
#         3
#     )
#   WHERE
#     RN = 1
# ),
# CLICK_STREAM_DEVICE AS (
#   SELECT
#     CUSTOMER_ID,
#     DEVICE_TYPE
#   FROM
#     (
#       SELECT
#         CUSTOMER_ID,
#         DEVICE_TYPE,
#         COUNT(DISTINCT EVENT_ID) AS EVENTS,
#         ROW_NUMBER() OVER (PARTITION BY CUSTOMER_ID ORDER BY COUNT(DISTINCT EVENT_ID) DESC) AS RN
#       FROM
#         ECOMMERCE.SILVER.CLICKSTREAM
#       WHERE CUSTOMER_ID <> -1
#       GROUP BY
#         1,
#         2
#     )
#   WHERE
#     RN = 1
# ),
# REVENUE_RANK AS (

#     SELECT
#     percentile_cont(0.33) WITHIN GROUP (ORDER BY TOTAL_SPEND) AS P33,
#     percentile_cont(0.66) WITHIN GROUP (ORDER BY TOTAL_SPEND) AS P66
# FROM SALES_KPIS

# )
# SELECT
#   A.*,
#   B.TOTAL_SPEND,
#   B.TOTAL_ORDERS,
#   B.TOTAL_QUANTITY,
#   B.TOTAL_SPEND / NULLIF(B.TOTAL_ORDERS, 0) AS AVERAGE_ORDER_VALUE,
#   C.PRODUCT_NAME AS PREFERRED_CATEGORY,
#   D.DEVICE_TYPE AS PREFERRED_DEVICE,
#   CASE WHEN B.TOTAL_SPEND < (SELECT P33 FROM REVENUE_RANK) THEN 'LOW'
#        WHEN B.TOTAL_SPEND BETWEEN (SELECT P33 FROM REVENUE_RANK) AND (SELECT P66 FROM REVENUE_RANK) THEN 'MEDIUM'
#        WHEN B.TOTAL_SPEND > (SELECT P66 FROM REVENUE_RANK)    THEN 'HIGH'
#        ELSE NULL END AS CUSTOMER_SEGMENT
# FROM
#   CUSTOMER_PUCHASE_DATE A
#     INNER JOIN SALES_KPIS B
#       ON A.CUSTOMER_ID = B.CUSTOMER_ID
#     INNER JOIN PREFERRED_PRODUCTS C
#       ON A.CUSTOMER_ID = C.CUSTOMER_ID
#     LEFT JOIN CLICK_STREAM_DEVICE D
#       ON A.CUSTOMER_ID = D.CUSTOMER_ID;

## Loading libraries for future usage

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


In [0]:
customers = spark.table("ECOMMERCE.SILVER.CUSTOMERS")
orders = spark.table("ECOMMERCE.SILVER.ORDERS")
order_items = spark.table("ECOMMERCE.SILVER.ORDER_ITEMS")
products = spark.table("ECOMMERCE.SILVER.PRODUCTS")
clickstream = spark.table("ECOMMERCE.SILVER.CLICKSTREAM")


In [0]:
customer_purchase_date = (
    customers.alias("A")
    .join(orders.alias("B"), F.col("A.CUSTOMER_ID") == F.col("B.CUSTOMER_ID"), "inner")
    .groupBy(F.col("A.CUSTOMER_ID"), F.col("A.FIRST_NAME"), F.col("A.LAST_NAME"))
    .agg(
        F.min("B.ORDER_DATE_UPDATED").alias("FIRST_ORDER_DATE"),
        F.max("B.ORDER_DATE_UPDATED").alias("LAST_ORDER_DATE"),
    )
    .withColumn(
        "DAYS_SINCE_LAST_ORDER", F.datediff(F.current_date(), F.col("LAST_ORDER_DATE"))
    )
)

In [0]:
sales_kpis = (
    orders.alias("A")
    .join(order_items.alias("B"), F.col("A.ORDER_ID") == F.col("B.ORDER_ID"), "inner")
    .groupBy(F.col("A.CUSTOMER_ID"))
    .agg(
        F.sum(
            F.when(
                F.col("A.ORDER_STATUS") == "DELIVERED", F.col("B.NET_AMOUNT")
            ).otherwise(0)
        ).alias("TOTAL_SPEND"),
        F.countDistinct(
            F.when(F.col("A.ORDER_STATUS") == "DELIVERED", F.col("A.ORDER_ID"))
        ).alias("TOTAL_ORDERS"),
        F.sum(
            F.when(
                F.col("A.ORDER_STATUS") == "DELIVERED", F.col("B.QUANTITY")
            ).otherwise(0)
        ).alias("TOTAL_QUANTITY"),
    )
)

In [0]:
revenue_rank = sales_kpis.agg(
    F.percentile_approx("TOTAL_SPEND", 0.33).alias("P33"),
    F.percentile_approx("TOTAL_SPEND", 0.66).alias("P66"),
)

In [0]:
preferred_product_base = (
    orders.alias("A")
    .join(order_items.alias("B"), F.col("A.ORDER_ID") == F.col("B.ORDER_ID"), "inner")
    .join(products.alias("C"), F.col("B.PRODUCT_ID") == F.col("C.PRODUCT_ID"), "inner")
    .filter(F.col("A.ORDER_STATUS") == "DELIVERED")
    .groupBy(F.col("A.CUSTOMER_ID"), F.col("B.PRODUCT_ID"), F.col("C.PRODUCT_NAME"))
    .agg(F.countDistinct("A.ORDER_ID").alias("PURCHASE_FREQUENCY"))
)

In [0]:
product_window = Window.partitionBy("CUSTOMER_ID").orderBy(
    F.col("PURCHASE_FREQUENCY").desc()
)

preferred_products = (
    preferred_product_base.withColumn("RN", F.row_number().over(product_window))
    .filter(F.col("RN") == 1)
    .select("CUSTOMER_ID", "PRODUCT_NAME", "PURCHASE_FREQUENCY")
)

In [0]:
clickstream_device_base = (
    clickstream.filter(F.col("CUSTOMER_ID") != -1)
    .groupBy("CUSTOMER_ID", "DEVICE_TYPE")
    .agg(F.countDistinct("EVENT_ID").alias("EVENTS"))
)


# Rank devices within each customer
device_window = Window.partitionBy("CUSTOMER_ID").orderBy(F.col("EVENTS").desc())

click_stream_device = (
    clickstream_device_base.withColumn("RN", F.row_number().over(device_window))
    .filter(F.col("RN") == 1)
    .select("CUSTOMER_ID", "DEVICE_TYPE")
)

In [0]:
final_df = (
    customer_purchase_date.alias("A")
    .join(
        sales_kpis.alias("B"), F.col("A.CUSTOMER_ID") == F.col("B.CUSTOMER_ID"), "inner"
    )
    .join(
        preferred_products.alias("C"),
        F.col("A.CUSTOMER_ID") == F.col("C.CUSTOMER_ID"),
        "inner",
    )
    .join(
        click_stream_device.alias("D"),
        F.col("A.CUSTOMER_ID") == F.col("D.CUSTOMER_ID"),
        "left",
    )
    # Bring P33 and P66 to every customer
    .crossJoin(revenue_rank)
    .select(
        F.col("A.*"),
        F.col("B.TOTAL_SPEND"),
        F.col("B.TOTAL_ORDERS"),
        F.col("B.TOTAL_QUANTITY"),
        # Average Order Value
        (
            F.col("B.TOTAL_SPEND")
            / F.when(F.col("B.TOTAL_ORDERS") != 0, F.col("B.TOTAL_ORDERS"))
        ).alias("AVERAGE_ORDER_VALUE"),
        F.col("C.PRODUCT_NAME").alias("PREFERRED_CATEGORY"),
        F.col("D.DEVICE_TYPE").alias("PREFERRED_DEVICE"),
        # Customer Segment
        F.when(F.col("B.TOTAL_SPEND") < F.col("P33"), "LOW")
        .when(F.col("B.TOTAL_SPEND").between(F.col("P33"), F.col("P66")), "MEDIUM")
        .when(F.col("B.TOTAL_SPEND") > F.col("P66"), "HIGH")
        .otherwise(None)
        .alias("CUSTOMER_SEGMENT"),
    )
)

In [0]:
display(final_df)

In [0]:
final_df.write.format("delta").mode("APPEND").saveAsTable(
    "ecommerce.gold.customer_360"
)

In [0]:
%sql
SELECT * FROM ECOMMERCE.GOLD.customer_360 LIMIT 10;